# Held-out test-set validation

This notebook replaces the archived checkpoint-validation notebook. Select one or more new experiment directories below; the saved `experiment_config.json` is loaded rather than retyped, so checkpoint architecture and held-out-data settings always match. All selected experiments must have identical configurations before their results can be compared.

It reconstructs final-stage replicas through the shared factory and evaluates them on the same untouched GTSRB test partition. Test predictions are used only in memory for supplemental bootstrap analysis and are not serialized. Do not use these results to tune training hyperparameters.

In [ ]:
! git clone "https://github.com/ddimpfel/JHU_IS_26.git"

In [ ]:
import os
os.chdir("/content/JHU_IS_26")
! pwd

In [ ]:
! pip install -r requirements.txt

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

from init_experiment import (
    ExperimentConfig,
    add_model_component_columns,
    bootstrap_performance_diff,
    build_component_delta_table,
    build_gtsrb_data,
    evaluate_checkpoints,
    normalize_backbone_label,
    prepare_notebook_runtime,
    run_paired_component_tests,
    summarize_component_performance,
)

# Add the exact experiment directories to validate. Each must contain
# `experiment_config.json`, `progress.json`, and final `models/` checkpoints.
RESULTS_NAMESPACE = 'test_validation'
EXPERIMENT_DIRS: list[Path] = [
    # Path('results/baseline_experiments/baseline_cil_protocol'),
    # Path('results/joint_embedding_experiments/joint_embedding_cil_protocol'),
]
CHECKPOINT_GLOB = '*__seed-*.pth'  # Final-stage replicas only.
BOOTSTRAP_RESAMPLES = 1_000

runtime = prepare_notebook_runtime(RESULTS_NAMESPACE)
if not EXPERIMENT_DIRS:
    raise ValueError(
        'Set EXPERIMENT_DIRS to one or more new experiment directories before evaluating checkpoints.'
    )

experiment_dirs: list[Path] = [
    (path if path.is_absolute() else runtime.project_dir / path).resolve()
    for path in EXPERIMENT_DIRS
]
config_paths = [path / 'experiment_config.json' for path in experiment_dirs]
missing_configs = [path for path in config_paths if not path.is_file()]
if missing_configs:
    raise FileNotFoundError(f'Missing experiment configuration(s): {missing_configs}')

configs = [ExperimentConfig.load(path) for path in config_paths]
config = configs[0]
if any(candidate.to_dict() != config.to_dict() for candidate in configs[1:]):
    raise ValueError('All selected experiment directories must use identical ExperimentConfig values.')

data = build_gtsrb_data(config)
MODELS_DIRS: list[Path] = [path / 'models' for path in experiment_dirs]
TEST_RESULTS_DIR = runtime.results_dir / 'test_results'
TEST_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

display(pd.DataFrame([config.to_dict()]))

In [ ]:
checkpoint_paths = sorted(
    (checkpoint for models_dir in MODELS_DIRS for checkpoint in models_dir.glob(CHECKPOINT_GLOB)),
    key=lambda path: (path.parent.parent.name, path.name),
)
if not checkpoint_paths:
    raise FileNotFoundError(
        f'No checkpoints matching {CHECKPOINT_GLOB!r} were found in the selected experiment directories.'
    )

pd.DataFrame({
    'Experiment': [path.parent.parent.name for path in checkpoint_paths],
    'Checkpoint': [path.name for path in checkpoint_paths],
})

In [ ]:
test_results_df, loaded_models = evaluate_checkpoints(checkpoint_paths, data, TEST_RESULTS_DIR)
test_results_df = add_model_component_columns(test_results_df)
test_results_df['Backbone Group'] = test_results_df['Generalist'].map(normalize_backbone_label)
test_results_df['Joint Embedding'] = test_results_df['Backbone Group'].str.startswith('JE ').map({True: 'JE', False: 'No JE'})

summary_columns = [
    'Model', 'Final Seed', 'Backbone Group', 'Router', 'Expert', 'Test Loss',
    'Test Macro F1', 'Test Micro F1', 'Test Weighted F1',
    'Test ECE', 'Test Router Entropy', 'Test Expected Expert Calls',
    'Test Cost Proxy', 'Num Parameters',
]
summary_df = test_results_df.loc[:, [column for column in summary_columns if column in test_results_df.columns]]
display(summary_df.sort_values(['Test Micro F1', 'Test Macro F1'], ascending=False).reset_index(drop=True))

In [ ]:
test_metrics = [
    metric for metric in ['Test Macro F1', 'Test Micro F1', 'Test Weighted F1', 'Test ECE',
                          'Test Router Entropy', 'Test Expected Expert Calls',
                          'Test Cost Proxy', 'Num Parameters']
    if metric in test_results_df.columns
]
backbone_summary_df = summarize_component_performance(test_results_df, 'Backbone Group', test_metrics)
display(backbone_summary_df)

backbone_order = ['MobileNet Large', 'ConvNeXt Tiny', 'JE MobileNet Large', 'JE ConvNeXt Tiny']
available_backbones = set(test_results_df['Backbone Group'].dropna())
comparisons = [name for name in backbone_order if name in available_backbones and name != 'MobileNet Large']
paired_metrics = [metric for metric in ['Test Macro F1', 'Test Micro F1', 'Test Weighted F1', 'Test ECE'] if metric in test_results_df.columns]
backbone_delta_df = build_component_delta_table(
    test_results_df, 'Backbone Group', 'MobileNet Large', comparisons, ['Router', 'Expert', 'Final Seed'], paired_metrics
)
backbone_significance_df = run_paired_component_tests(
    test_results_df, 'Backbone Group', 'MobileNet Large', comparisons, ['Router', 'Expert', 'Final Seed'], paired_metrics
)
display(backbone_delta_df)
display(backbone_significance_df)

In [ ]:
print('Bootstrap intervals resample held-out examples conditional on the trained replicas; they are supplemental and are not independent training-replica inference.')
if len(loaded_models) >= 2:
    bootstrap_results, bootstrap_model_performance = bootstrap_performance_diff(
        data.test_loader(),
        loaded_models,
        device=config.resolved_device,
        resamples=BOOTSTRAP_RESAMPLES,
    )
    display(pd.DataFrame(bootstrap_model_performance))
    display(pd.DataFrame(bootstrap_results))
else:
    print('At least two successfully reconstructed checkpoints are required for pairwise bootstrap comparisons.')

In [ ]:
if {'Test Micro F1', 'Backbone Group', 'Router'}.issubset(test_results_df.columns):
    sns.set_theme(style='whitegrid')
    plt.figure(figsize=(12, 6))
    plot_df = test_results_df.sort_values('Test Micro F1', ascending=False)
    sns.barplot(data=plot_df, x='Test Micro F1', y='Model', hue='Backbone Group', errorbar=None)
    plt.title('Held-out Test Micro F1 by Final Model Replica')
    plt.tight_layout()
    plt.show()